# Boston House Classification

**Tabular Regression Project** — Predict median house values in Boston suburbs.

Models: CatBoost · LightGBM · XGBoost · FLAML · LazyPredict
Dataset: Boston Housing (506 rows × 14 features)
Target: `medv` (median value in $1000s)

## 2 · Project Overview

Despite the folder name containing 'Classification', this is a **regression** task. We predict the **median value of owner-occupied homes** (`medv`) in Boston suburbs using 13 socioeconomic and structural features.

The Boston Housing dataset is one of the most classic ML datasets, ideal for learning regression fundamentals on a small, well-understood dataset.

## 3 · Learning Objectives

1. Work with the classic Boston Housing dataset.
2. Understand 13 diverse features (crime rate, room count, tax rate, etc.).
3. Build and compare baseline and boosting models.
4. Use LazyPredict and FLAML for model selection.
5. Perform residual analysis to identify model weaknesses.

## 4 · Problem Statement

Given 13 features describing a Boston suburb (crime rate, number of rooms, pupil-teacher ratio, etc.), predict the **median house value** (`medv`). This helps urban planners and real-estate analysts understand price drivers.

## 5 · Why This Project Matters

- **Real estate valuation** is a fundamental regression application.
- The dataset exposes how socioeconomic factors interact with housing prices.
- At 506 rows, it's small enough for rapid experimentation.
- Understanding feature importance here maps directly to policy-relevant insights.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 506 |
| **Features** | crim, zn, indus, chas, nox, rm, age, dis, rad, tax, ptratio, black, lstat |
| **Target** | medv (median value in $1000s) |
| **Missing** | None |
| **Note** | Loaded from sibling folder's `Boston Dataset.csv` |

## 7 · Dataset Source and License Notes

- **Source**: UCI ML Repository / Carnegie Mellon StatLib.
- **License**: Public domain for research and education.
- **Note**: This dataset has been criticized for the `black` variable. We use it here for educational ML purposes only.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install_if_missing(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

for pkg, imp in [('catboost', None), ('lightgbm', None), ('xgboost', None),
                 ('flaml', None), ('lazypredict', None)]:
    _install_if_missing(pkg, imp)

print('All packages ready.')

## 9 · Imports

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
try:
    from sklearn.metrics import root_mean_squared_error
except ImportError:
    from sklearn.metrics import mean_squared_error
    def root_mean_squared_error(y_true, y_pred): return mean_squared_error(y_true, y_pred, squared=False)

warnings.filterwarnings('ignore')
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
print('Imports complete.')

## 10 · Configuration / Constants

In [ ]:
SEED = 42
TEST_SIZE = 0.2
TARGET = 'medv'
np.random.seed(SEED)

## 11 · Dataset Download or Loading

In [ ]:
# Try local first, then sibling folder
DATA_PATH = os.path.join(SAVE_DIR, 'Boston Dataset.csv')
if not os.path.exists(DATA_PATH):
    DATA_PATH = os.path.join(SAVE_DIR, '..', 'Boston Housing Prediction Analysis', 'Boston Dataset.csv')
assert os.path.exists(DATA_PATH), f'Dataset not found at {DATA_PATH}'
df = pd.read_csv(DATA_PATH)
# Drop unnamed index column if present
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])
print(f'Loaded: {df.shape}')
df.head()

## 12 · Data Validation Checks

In [ ]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'Target range: [{df[TARGET].min()}, {df[TARGET].max()}]')
print(f'Target mean: {df[TARGET].mean():.2f}')

## 13 · Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0,0].scatter(df['rm'], df[TARGET], alpha=0.5, s=10)
axes[0,0].set_xlabel('rm (avg rooms)'); axes[0,0].set_ylabel(TARGET)
axes[0,0].set_title('Rooms vs medv')

axes[0,1].scatter(df['lstat'], df[TARGET], alpha=0.5, s=10)
axes[0,1].set_xlabel('lstat (% lower status)'); axes[0,1].set_ylabel(TARGET)
axes[0,1].set_title('lstat vs medv')

df[TARGET].hist(bins=30, ax=axes[1,0], edgecolor='black')
axes[1,0].set_title('medv Distribution')

corr = df.corr()[TARGET].drop(TARGET).sort_values()
corr.plot(kind='barh', ax=axes[1,1])
axes[1,1].set_title('Feature Correlations with medv')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## 14 · Target Analysis

In [ ]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')
print(f'Values at 50.0 (cap): {(df[TARGET] == 50.0).sum()}')
print('Note: medv is capped at 50.0 — this is censored data.')

## 15 · Train / Validation / Test Split Strategy

80/20 random split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 16 · Preprocessing Strategy

All features are numeric. No encoding or scaling needed for tree-based models. Linear regression baseline will work on raw features.

In [ ]:
print('Feature dtypes:')
print(X_train.dtypes)
print(f'\nAll numeric: {X_train.select_dtypes(include="number").shape[1] == X_train.shape[1]}')

## 17 · Feature Engineering

The dataset is already well-curated. We add one ratio feature.

In [ ]:
for df_part in [X_train, X_test]:
    df_part['rooms_per_dwelling'] = df_part['rm'] / (df_part['age'] + 1) * 100

print(f'Features: {X_train.shape[1]}')

## 18 · Baseline Model

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_base = baseline.predict(X_test)

baseline_rmse = root_mean_squared_error(y_test, y_pred_base)
baseline_mae = mean_absolute_error(y_test, y_pred_base)
baseline_r2 = r2_score(y_test, y_pred_base)

print(f'Baseline Linear Regression:')
print(f'  RMSE: {baseline_rmse:.2f}')
print(f'  MAE:  {baseline_mae:.2f}')
print(f'  R²:   {baseline_r2:.4f}')

## 19 · LazyPredict Benchmark

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True, random_state=SEED)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)
print(lazy_models.head(15))

## 20 · FLAML AutoML Run

In [ ]:
try:
    from flaml import AutoML
    flaml_model = AutoML()
    flaml_model.fit(
        X_train, y_train,
        task='regression',
        time_budget=60,
        metric='rmse',
        seed=SEED,
        verbose=0
    )
    y_pred_flaml = flaml_model.predict(X_test)
    flaml_rmse = root_mean_squared_error(y_test, y_pred_flaml)
    flaml_mae = mean_absolute_error(y_test, y_pred_flaml)
    flaml_r2 = r2_score(y_test, y_pred_flaml)
    print(f'FLAML best: {flaml_model.best_estimator}')
    print(f'  RMSE: {flaml_rmse:.2f}')
    print(f'  MAE:  {flaml_mae:.2f}')
    print(f'  R²:   {flaml_r2:.4f}')
except Exception as e:
    print(f'FLAML failed: {e}')
    flaml_rmse = flaml_mae = flaml_r2 = None

## 21 · Additional Best-Library Workflow (CatBoost + LightGBM + XGBoost)

In [ ]:
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

models = {
    'CatBoost': CatBoostRegressor(iterations=500, learning_rate=0.1, depth=6,
                                   random_seed=SEED, verbose=0),
    'LightGBM': LGBMRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                              random_state=SEED, verbose=-1),
    'XGBoost': XGBRegressor(n_estimators=500, learning_rate=0.1, max_depth=6,
                            random_state=SEED, verbosity=0)
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results[name] = {
        'RMSE': root_mean_squared_error(y_test, preds),
        'MAE': mean_absolute_error(y_test, preds),
        'R2': r2_score(y_test, preds)
    }
    print(f'{name}: RMSE={results[name]["RMSE"]:.2f}, MAE={results[name]["MAE"]:.2f}, R²={results[name]["R2"]:.4f}')

## 22 · Top 3 Model Selection

In [ ]:
all_results = {}
all_results['Baseline_LR'] = {'RMSE': baseline_rmse, 'MAE': baseline_mae, 'R2': baseline_r2}
if flaml_r2 is not None:
    all_results['FLAML'] = {'RMSE': flaml_rmse, 'MAE': flaml_mae, 'R2': flaml_r2}
all_results.update(results)

ranked = sorted(all_results.items(), key=lambda x: x[1]['RMSE'])
print('All models ranked by RMSE:')
for name, m in ranked:
    print(f"  {name:20s}  RMSE={m['RMSE']:.2f}  MAE={m['MAE']:.2f}  R²={m['R2']:.4f}")

top3_names = [r[0] for r in ranked[:3]]
print(f'\nTop 3: {top3_names}')

## 23 · Final Training and Evaluation of Top 3

In [ ]:
print('Top 3 Final Results:')
print('=' * 60)
for name in top3_names:
    m = all_results[name]
    print(f"{name}:")
    print(f"  RMSE: {m['RMSE']:.2f}")
    print(f"  MAE:  {m['MAE']:.2f}")
    print(f"  R²:   {m['R2']:.4f}")
    print()

## 24 · Error Analysis

In [ ]:
best_name = top3_names[0]
if best_name in models:
    best_model = models[best_name]
else:
    best_model = models['CatBoost']

y_pred_best = best_model.predict(X_test)
residuals = y_test.values - y_pred_best

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].scatter(y_pred_best, residuals, alpha=0.5)
axes[0].axhline(0, color='r', linestyle='--')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted')

axes[1].hist(residuals, bins=30, edgecolor='black')
axes[1].set_title('Residual Distribution')

axes[2].scatter(y_test, y_pred_best, alpha=0.5)
axes[2].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[2].set_xlabel('Actual'); axes[2].set_ylabel('Predicted')
axes[2].set_title('Actual vs Predicted')

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## 25 · Interpretation and Business Insight

- **rm** (number of rooms) and **lstat** (% lower-status population) are the two strongest predictors.
- Areas with more rooms and lower % lower-status population have higher median values.
- The capped values at medv=50 create prediction artifacts — models tend to under-predict for expensive suburbs.
- These insights align with real-world real estate: space and neighborhood demographics drive prices.

## 26 · Limitations

- medv is censored at 50.0 — true top-end values are unknown.
- The `black` variable raises ethical concerns and would not be used in modern fair-lending models.
- Only 506 rows limits model complexity.
- Data is from the 1970s Census — today's Boston is very different.

## 27 · How to Improve This Project

1. Remove censored observations (medv=50) or model censoring explicitly.
2. Try polynomial/interaction features for linear regression.
3. Use cross-validation instead of a single train/test split.
4. Compare with the California Housing dataset (larger, uncensored).

## 28 · Production Considerations

- Would need current data — 1970s Census data is obsolete.
- Fair lending regulations prohibit certain features (race-related).
- Need regular retraining as neighborhoods change.
- Spatial autocorrelation: nearby homes have correlated prices.

## 29 · Common Mistakes

1. **Ignoring the medv cap at 50**: Distorts RMSE and R².
2. **Using `black` without ethical consideration**: Problematic in production.
3. **Not checking for multicollinearity**: rad and tax are highly correlated.
4. **Treating chas as continuous**: It's binary (0/1).

## 30 · Mini Challenge / Exercises

1. Remove all rows where medv=50 and retrain — does R² improve?
2. Add polynomial features for rm and lstat.
3. Use SHAP to explain the 5 most expensive predictions.
4. Try ridge/lasso regression and compare with OLS.

## 31 · Final Summary / Key Takeaways

- Gradient-boosting models outperform linear regression on Boston Housing.
- Number of rooms (rm) and lower-status percentage (lstat) dominate predictions.
- The medv cap at 50 is a data quality issue that affects model evaluation.
- Despite its age, this dataset teaches fundamental regression concepts well.

In [ ]:
metrics = {}
for name in top3_names:
    metrics[name] = all_results[name]
metrics['Baseline_LR'] = all_results['Baseline_LR']

with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metrics saved to metrics.json')
print('\nNotebook complete.')